# Adding CAD objects to the Isaac Sim cell

**Time:** 2–3 hours (CAD export + Isaac Sim on Brev)  
**Prerequisites:** Notebooks 00–03 (or equivalent), `bash scripts/setup-dev.sh`, Isaac configured (`ISAAC` set).

Notebook **03** shows how existing cubes and distractors are *configured*. This notebook covers bringing in **new geometry from CAD** (STEP, STL, etc.) as USD and wiring it into the scene like the starter assets.

## Learning objectives

- Export or convert CAD to **USD** with a clear **origin** and **units**
- Place assets under `fanuc_er4ia_sim/assets/` and match the starter’s inch-CAD + metre-stage pattern
- Add **physics colliders** (simple proxies) separate from high-poly visuals
- Register layout in `scene_config.py` and scene build code in `cell.py`
- Rebuild the scene, recapture, and understand impact on perception/planning

## Checkpoint

Your new mesh appears in a headless `build_scene.py --save` USD, rests on the mat under gravity, and does not break the six-cube hackathon task unless you intentionally change task semantics.

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

def find_root() -> Path:
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "fanuc_er4ia_sim" / "cell.py").is_file():
            return p
    raise FileNotFoundError("Could not find repository root")

ROOT = find_root()
SIM = ROOT / "fanuc_er4ia_sim"
USD_DIR = SIM / "assets" / "usd_assets"
sys.path.insert(0, str(SIM))
import scene_config as C

print("Repository:", ROOT)
print("Bundled USD assets:", sorted(p.name for p in USD_DIR.glob("*.usda")))

## 1. How the starter loads CAD-based USD

Task visuals live in [`fanuc_er4ia_sim/assets/usd_assets/`](../../fanuc_er4ia_sim/assets/usd_assets/). Each file documents CAD source and **origin** in its header (open `cube.usda` in a text editor).

| Asset | CAD nominal (file) | Sim size (`scene_config`) | Origin |
|-------|--------------------|---------------------------|--------|
| `cube.usda` | 16 mm edge | `CUBE_EDGE` = 10 mm (hackathon fit) | Bottom-face centre |
| `vial.usda` | 203 mm long | `VIAL_LENGTH` scaled | Bottom centre |
| `vial_cap.usda` | 15.7 mm tall | `CAP_HEIGHT` scaled | Flange underside |

Pipeline in code:

1. **`scene_config.py`** — table XY positions, masses, sim dimensions in **metres**.
2. **`cell.add_sort_objects()`** — reference USD, `scale_cad_visual()` for photo/sim fit, **invisible** primitive colliders, `make_dynamic()`.

Isaac Sim also applies **inch→metre** (`metersPerUnit = 0.0254`) on referenced CAD USD; do not add a second unit scale on top unless you know the file’s units.

In [ ]:
cube_usda = USD_DIR / "cube.usda"
head = cube_usda.read_text(encoding="utf-8", errors="replace").splitlines()[:15]
print("\n".join(head))

# Photo/sim fit scale used in cell.py for cubes (see add_sort_objects):
cad_nominal_edge_m = 0.016  # metres at 1:1 after USD inch resolver for this asset
sim_edge = C.CUBE_EDGE
scale_factor = sim_edge / cad_nominal_edge_m
print(f"\nCube scale tuple (x,y,z): ({scale_factor},) * 3  # sim {sim_edge} m vs CAD nominal {cad_nominal_edge_m} m")

## 2. CAD → USD export (choose one path)

The repo does **not** ship an exporter; bundled USD was produced offline (`generator = usd_assets/export_assets.py` in layer metadata). Pick a path your team supports:

### A. Isaac Sim / Omniverse (recommended on Brev)

1. Import STEP/STL/OBJ via **Content Browser** or **Create → Import**.
2. Set **stage metres** and place the part at the origin you want on the table (usually **bottom centre** on Z=0).
3. Apply materials; keep triangle count reasonable for RTX captures.
4. **File → Export** (or collect) to `fanuc_er4ia_sim/assets/usd_assets/my_part.usda` (or `.usd`).
5. Note in the layer `doc` string: CAD filename, measured bounding box, and origin convention.

### B. Blender → USD

Export with **Z up**, document scale (metres), verify bounding box in Blender before export.

### C. CAD tool native USD (if available)

Same rules: explicit units, single root transform, document origin.

**Team convention:** store team-specific meshes as `assets/usd_assets/<name>.usda` and commit only if license/size allows (keep files small; no vendor confidential meshes in public repos).

## 3. Pre-flight checklist for a new object

- [ ] **Origin** matches how you translate in `cell.py` (cubes: bottom centre on mat `Z = TABLE_TOP_Z + SURFACE_DZ`).
- [ ] **Sim dimensions** in `scene_config.py` (metres) match tape measure or CAD drawing, not the raw file name.
- [ ] **Collider** is a simple box/cylinder/capsule; visuals can be high-poly mesh.
- [ ] **Mass** is plausible (`make_dynamic`); too light → jitter; too heavy → unrealistic sliding.
- [ ] **Clearance** from cube stack site `CUBE_STACK_XY` and gripper hover paths (host tests check IK for cube layout).
- [ ] If you change layout/camera/size: **new capture** in an empty `runs/` directory (see `SETUP.md`).
- [ ] If shipping a hackathon zip: add the file to `package_hackathon.py` allowlist.

**Distractor vs task object:** extra CAD props should stay out of `plan_sort` cube picks unless you extend prompts and planning on purpose.

## 4. Wire-up pattern (config + scene code)

Below is the **cube** pattern from `cell.add_sort_objects` — adapt names and collider shape for your part.

**Step A — `scene_config.py`** (example custom distractor):

```python
# Example: small puck distractor (metres)
MY_PUCK_RADIUS = 0.012
MY_PUCK_HEIGHT = 0.008
MY_PUCK_MASS = 0.003
MY_PUCKS = [(0.50, -0.05)]  # table XY
CAD_PUCK_DIAMETER_NOMINAL = 0.025  # metres after USD unit resolve at scale 1
```

**Step B — `cell.py`** inside or beside `add_sort_objects`:

```python
for i, (x, y) in enumerate(C.MY_PUCKS):
    g = UsdGeom.Xform.Define(stage, f"/World/MyPuck{i}")
    UsdGeom.Xformable(g.GetPrim()).AddTranslateOp().Set(Gf.Vec3d(x, y, Z))
    vis_path = f"/World/MyPuck{i}/Visual"
    add_reference_to_stage(
        usd_path=os.path.join(here, "assets/usd_assets/my_puck.usda"),
        prim_path=vis_path,
    )
    fit = C.MY_PUCK_RADIUS * 2 / C.CAD_PUCK_DIAMETER_NOMINAL
    scale_cad_visual(stage, vis_path, (fit, fit, C.MY_PUCK_HEIGHT / C.CAD_PUCK_HEIGHT_NOMINAL))
    # Cylinder collider, invisible, same as caps/cubes...
    make_dynamic(g.GetPrim(), C.MY_PUCK_MASS, collide_self=False)
```

Call your helper from `build()` when `with_objects=True` (same gate as `add_sort_objects`).

In [ ]:
# Inspect separation from existing cube centres (placement sanity check)
import math

def min_dist_to_cubes(xy):
    return min(math.dist(xy, (x, y)) for x, y, _ in C.SORT_CUBES)

candidate = (0.50, -0.05)
print("Candidate puck XY:", candidate)
print("Distance to nearest cube centre:", f"{min_dist_to_cubes(candidate):.3f} m")
print("Distance to stack site:", f"{math.dist(candidate, C.CUBE_STACK_XY):.3f} m")
if min_dist_to_cubes(candidate) < 0.065:
    print("Warning: closer than typical cube spacing — adjust XY.")

## 5. Validate in Isaac Sim

After adding your USD file and code changes, rebuild on Brev (bash cells assume `ISAAC` is set).

In [ ]:
isaac = os.environ.get("ISAAC")
if not isaac:
    print("Set ISAAC to your Isaac Sim install, then re-run this section.")
else:
    out_usd = SIM / "runs" / "cad_object_preview.usd"
    out_usd.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        f"{isaac}/python.sh",
        str(SIM / "build_scene.py"),
        "--headless",
        "--save",
        str(out_usd),
    ]
    print(" ".join(cmd))
    # Uncomment to run on Brev:
    # subprocess.run(cmd, check=True)
    print(f"When run, open {out_usd} in Isaac or inspect with usdview.")

### Interactive review (optional)

```bash
cd fanuc_er4ia_sim
"$ISAAC/python.sh" build_scene.py
```

Confirm: object sits on the white mat, does not interpenetrate the table, and does not block the table camera view of all six cubes.

### Still capture for perception

```bash
mkdir -p runs/my_cad_layout
"$ISAAC/python.sh" capture_scene.py --out runs/my_cad_layout
```

If you only added a **distractor**, the default `STACK_PROMPT` may ignore it — that is expected. To study novel shapes, extend notebook **06** (prompt / ICL) or add a dedicated detection class in `perceive.py` with team review.

## 6. Exercises

1. **Measure:** From CAD or USD metadata, fill a small table: file name, bounding box (mm), chosen origin, sim dimensions in `scene_config`.
2. **Export:** Produce `assets/usd_assets/team_distractor.usda` and verify `metersPerUnit` / up axis in the file header.
3. **Integrate:** Add one instance to the scene with a cylinder collider; run host tests (`bash scripts/setup-dev.sh` or unittest in `fanuc_er4ia_sim`).
4. **Capture:** Run `capture_scene.py` and confirm the object appears in `world_camera.png` without hiding cube silhouettes.
5. **Document:** In your team README, note license/source of CAD and whether the mesh is committed or downloaded separately.

## Troubleshooting

| Symptom | Likely cause |
|---------|----------------|
| Giant or tiny object | Double unit scale; wrong `scale_cad_visual` ratio |
| Floating or sunken | Wrong origin; wrong `Z` offset (mat surface `SURFACE_DZ`) |
| Object explodes / jitters | Collider mesh too complex; interpenetration at spawn — raise spawn Z slightly |
| Invisible in capture | Outside table camera frustum; check `TABLE_CAM` work area |
| Pipeline still passes but VLM confused | Distractor resembles cubes — change colour/shape or prompt |

## Cleanup

- Revert experimental layout before graded runs unless the team adopts it.
- Do not commit multi-megabyte CAD sources unless required; link to internal storage instead.